# 01 — Learning curves

Was the agent actually learning?  Plots episode return + agent-update
metrics (loss, entropy, KL, …) over the course of training, broken
out by phase (``train`` / ``inline_eval`` / ``eval``).

In [ ]:
import sys, pathlib
# Make ``utils.py`` importable whether the notebook is opened from
# project root or from analysis/.
_here = pathlib.Path.cwd()
for cand in (_here, _here / "analysis", _here.parent):
    if (cand / "utils.py").exists():
        sys.path.insert(0, str(cand))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    list_runs, latest_run, load_run, load_runs,
    group_columns, group_columns_by_prefix, strip_prefix, rolling_mean,
)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

In [ ]:
# Pick a run.  ``latest_run()`` returns the most recently modified
# experiment in ``reports/``.  Override ``EXP_ID = "..."`` to analyse
# a specific one.
EXP_ID = latest_run()
print("Analysing:", EXP_ID)

ep, st, cfg = load_run(EXP_ID)
print(f"  episode_metrics.csv: shape={ep.shape}")
print(f"  step_metrics.csv:    shape={st.shape if st is not None else 'absent'}")

## Episode return

Raw + rolling-window average for the training phase.

In [ ]:
train = ep[ep["phase"] == "train"].copy()
train["return_smooth"] = rolling_mean(train["return"], window=max(5, len(train)//30 or 1))

fig, ax = plt.subplots()
ax.plot(train["episode"], train["return"], lw=0.5, color="C0", alpha=0.4, label="raw")
ax.plot(train["episode"], train["return_smooth"], lw=2, color="C0", label="rolling avg")

# Overlay eval points if present
for phase, marker, color in [("inline_eval", "o", "C1"), ("eval", "s", "C2")]:
    sub = ep[ep["phase"] == phase]
    if not sub.empty:
        ax.scatter(sub["episode"], sub["return"], s=30, marker=marker, color=color, label=phase, zorder=3)

ax.set(xlabel="episode", ylabel="episode return", title="Episode return over training")
ax.legend()
plt.show()

## Agent update metrics (PPO)

`loss`, `entropy`, plus PPO-specific `pg_loss`, `vf_loss`, `approx_kl`,
`clip_fraction`, `lr` if recorded.

In [ ]:
update_cols = ["loss", "entropy"] + group_columns(train, "update")
update_cols = [c for c in update_cols if c in train.columns and train[c].notna().any()]

if update_cols:
    n = len(update_cols)
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(11, 2.6 * nrows), squeeze=False)
    for ax, col in zip(axes.flat, update_cols):
        ax.plot(train["episode"], train[col], lw=1)
        ax.set_title(col.split('/', 1)[-1])
        ax.set_xlabel("episode")
    for ax in axes.flat[len(update_cols):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print("No update metrics in this run (heuristic baseline?)")

## Train vs eval gap

If `inline_eval` is enabled, the gap between train and eval returns
is the standard "is the policy overfitting the latest rollout"
diagnostic.

In [ ]:
if "inline_eval" in ep["phase"].unique():
    fig, ax = plt.subplots()
    for phase, color in [("train", "C0"), ("inline_eval", "C1")]:
        sub = ep[ep["phase"] == phase].copy()
        sub["smooth"] = rolling_mean(sub["return"], window=max(3, len(sub)//30 or 1))
        ax.plot(sub["episode"], sub["smooth"], label=phase, color=color, lw=2)
    ax.set(xlabel="episode", ylabel="return (rolling avg)", title="Train vs inline-eval")
    ax.legend()
    plt.show()
else:
    print("This run has no inline_eval phase to compare against.")